# Engine and Optimizations

## Overview ##

This notebook presents a simple way to use the graph_rewrite library for optimizing query execution. We would use spannerlog to write the queries and generate their execution graphs, and a modified version of spannerlib's engine defined in this notebook to run the execution graphs, while profiling the query runtime metrics.

In [737]:
import pandas as pd
pd.set_option("mode.copy_on_write", True)
import numpy as np
from typing import no_type_check, Set, Sequence, Any,Optional,List,Callable,Dict,Union
import networkx as nx
import itertools
from collections import defaultdict
import sys
import os
from pathlib import Path
sys.path.append("../../spannerlib/spannerlib")

import time


from spannerlib.span import Span
from spannerlib.data_types import (
    Var, 
    FreeVar, 
    RelationDefinition, 
    Relation, 
    IEFunction,
    AGGFunction,
    IERelation, 
    Rule, 
    pretty
)
from spannerlib.ra import (
    _col_names,
    get_const,
    select,
    project,
    rename,
    union,
    intersection,
    difference,
    join,
    product,
    groupby,
    ie_map,
    merge_rows
)

from spannerlib.term_graph import graph_compose, merge_term_graphs_pair,rule_to_graph,add_relation,add_project_uniq_free_vars
from spannerlib.engine import Engine, IEFunction, AGGFunction
from spannerlib import get_magic_session,Span


from graph_rewrite import draw
from graph_rewrite import rewrite, rewrite_iter


import logging
logger = logging.getLogger(__name__)

## Utils ##

Define the engine (compute_node func) and its dependencies.

The engine is a simplified version of the Spannerlib engine, designed to operate on trees. It extends the original implementation by integrating profiling capabilities.


In [738]:
def schema_match(schema,expected,ignore_types=None):
    """checks if"""
    if len(schema) != len(expected):
        return False
    if ignore_types is None:
        ignore_types = []
    for x,y in zip(schema,expected):
        if x in ignore_types:
            continue
        if not issubclass(x,y):
            return False
    return True


def is_of_schema(relation,schema,ignore_types=None):
    """checks if a relation is of a given schema"""
    try:
        if len(relation) != len(schema):
            return False
        if ignore_types is None:
            ignore_types = []
        for x,y in zip(relation,schema):
            if type(x) in ignore_types:
                continue
            if not isinstance(x,y):
                return False
        return True
    except Exception as e:
        logger.error(f"Got Error when computing:\n"
                     f"is_of_scehma({relation},{schema})\n"
                     f"Error: {e}")
        raise e

def type_merge(type1,type2):
    if issubclass(type1,type2):
        return type1
    elif issubclass(type2,type1):
        return type2
    else:
        raise ValueError(f"Trying to merge types {type1},{type2}, types are incompatible")

def schema_merge(schema1,schema2):
    """merges two schemas, taking the stricter type between the two for each index"""
    if len(schema1) != len(schema2):
        raise ValueError(f"Trying to merge schemas {schema1},{schema2} schemas must be of the same length")
    
    new_schema = [type_merge(x,y) for x,y in zip(schema1,schema2)]
    return new_schema

In [739]:
import re
STRING_PATTERN = re.compile(r"^[^\r\n]+$")

def isFloat(s):  
   n = '0123456789.' 
   return (all(x in n for x in s) and s.count('.') == 1)  
 
def isInt(s):  
   n = '0123456789'    
   return all(x in n for x in s) 

def _infer_relation_schema(row) -> Sequence[type]: # Inferred type list of the given relation
    """
    Guess the relation type based on the data.
    We support both the actual types (e.g. 'Span'), and their string representation ( e.g. `"[0,8)"`).

    **@raise** ValueError: if there is a cell inside `row` of an illegal type.
    """
    relation_types = []
    for cell in row:
        if not isinstance(cell, str):
            relation_types.append(type(cell))
        elif isInt(cell):
            relation_types.append(int)
        elif isFloat(cell):
            relation_types.append(float)
        elif cell in ['True', 'False']:
            relation_types.append(bool)
        else:
            relation_types.append(str)
        
    return relation_types

In [740]:
class DB(dict):
    def __repr__(self):
        key_str=', '.join(self.keys())
        return f'DB({key_str})'

In [741]:
def _col_names(length):
    # these names wont conflict with logical variables since they must always start with Uppercase letters
    return [f'col_{i}' for i in range(length)]

In [ ]:
# some select theta functions

class equalConstTheta():
    def __init__(self,*pos_val_tuples):
        self.pos_val_tuples = pos_val_tuples
    def __call__(self,df):
        masks = [df.iloc[:,pos]==val for pos,val in self.pos_val_tuples]
        return pd.concat(masks,axis=1).all(axis=1)
    def __str__(self):
        return f'''Theta({', '.join([f'col_{pos}={val}' for pos,val in self.pos_val_tuples])})'''
    def __repr__(self):
        return str(self)
    def __eq__(self,other):
        if not isinstance(other,equalConstTheta):
            return False
        return self.pos_val_tuples == other.pos_val_tuples

class equalColTheta():
    def __init__(self,*col_pos_tuples):
        self.col_pos_tuples = col_pos_tuples

    def __call__(self,df):
        masks = [df.iloc[:,pos1]==df.iloc[:,pos2] for pos1,pos2 in self.col_pos_tuples]
        return pd.concat(masks,axis=1).all(axis=1)    
    def __str__(self):
        return f'''Theta({', '.join([f'col_{pos1}=col_{pos2}' for pos1,pos2 in self.col_pos_tuples])})'''
    def __repr__(self):
        return str(self)
    def __eq__(self,other):
        if not isinstance(other,equalColTheta):
            return False
        return self.col_pos_tuples == other.col_pos_tuples

In [743]:
def get_const(const_dict,**kwargs):
    return pd.DataFrame([const_dict])


def is_truthy(df):
    return df.shape==(1,0)

def is_falsy(df):
    return df.shape==(0,0)

In [744]:
def select(df,theta,schema,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    if callable(theta):
        return df[theta(df)]
    else:
        raise ValueError(f"theta must be callable, got {theta}")

def project(df,schema,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    return df[schema]
    
def rename(df,schema,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    
    df=df.copy()
    df.columns = schema
    return df

def intersection(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or df1.empty or df2.empty:
        return pd.DataFrame(columns=schema)
    return pd.merge(df1,df2,how='inner',on=list(df1.columns))

def difference(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or df1.empty or df2.empty:
        return pd.DataFrame(columns=schema)
    return pd.concat([df1,df2]).drop_duplicates(keep=False)


def product(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or df1.empty or df2.empty:
        return pd.DataFrame(columns=schema)
    return pd.merge(df1,df2,how='cross')

def join(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or is_falsy(df1) or is_falsy(df2):
        return pd.DataFrame(columns=schema)

    # if one of the dataframes is truthy, return the other
    # this solves the problem of joining with a constant
    if is_truthy(df1):
        return df2
    if is_truthy(df2):
        return df1

    cols1 = set(df1.columns)
    cols2 = set(df2.columns)
    on = cols1 & cols2
    # get only logical variables
    # on = [ col for col in on if isinstance(col,str) and col[0].isupper()]
    on = list(on)
    if len(on)==0:
        return pd.merge(df1,df2,how='cross')
    else:
        return pd.merge(df1,df2,how='inner',on=on)
    
def merge_rows(*dfs):
    return pd.DataFrame(
        set.union(*[set(df.itertuples(index=False,name=None)) for df in dfs])
    )


def union(*dfs,schema,**kwargs):
    # use numpy arrays to ignore column names
    non_empty_dfs = []
    for df in dfs:
        if df is not None and not df.empty:
            non_empty_dfs.append(df)
    if len(non_empty_dfs)==0:
        return pd.DataFrame(columns=schema)
    else:
        return rename(merge_rows(*non_empty_dfs),schema)
        # This line didnt work since drop duplicates doesnt work correctly on non primitive classes such as Spans
        # return pd.DataFrame(np.concatenate(non_empty_dfs,axis=0),columns=schema).drop_duplicates(ignore_index=True)
        
def groupby(df,schema,agg,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    
    # rename columns to numbers so that we can aggregate the same free var to multiple places
    uniq_cols_df = rename(df,schema=[i for i in range(len(schema))])

    groupby_cols = [i for i,agg_func in enumerate(agg) if agg_func is None]
    agg_by_cols = {i:agg_func for i,agg_func in enumerate(agg) if agg_func is not None}
    # a real groupby
    if len(groupby_cols)>0:
        return rename(
            project(
                uniq_cols_df.groupby(groupby_cols).agg(agg_by_cols).reset_index(),
                schema = uniq_cols_df.columns
                ),
            schema)
    # no group by vars, so aggs contain all columns and schema simply orders them
    else:

        # this conversion magic is caused by an inconsistency between series and dataframes aggs,
        # to enable using both function and str aliases we
        # we take each column, convert to a frame
        # aggregate it and then squeeze it to a series (which has a single value)
        # then feed that to the dataframe constructor
        return rename(
            pd.DataFrame({
                col:[uniq_cols_df[col].to_frame().agg(agg_by_cols[col]).squeeze()] for col in range(len(agg_by_cols))
            }),
            schema)

In [745]:
def coerce_tuple_like(name,func,input,output):
    if isinstance(output,(tuple,list)):
        return output
    else:
        logger.debug(f"IEFunction {name} with underlying function {func}\n"
                        f"returned a value that is not a tuple/list\n"
                        f"for input {input} -> {output}\n"
                        f"coercing to tuple")
        return (output,)

def assert_ie_schema(name,func,value,expected_schema,arity,input_or_output='input'):
    if callable(expected_schema):
        expected_schema = expected_schema(arity)
    if not is_of_schema(value,expected_schema):
        raise ValueError(
            f"IEFunction {name} with underlying function {func}\n"
            f"received an {input_or_output} value {value}(schema={pretty(_infer_relation_schema(value))})\n"
            f"but expected {pretty(expected_schema)}")

def assert_iterable(name,func,input,output):
    try:
        out_iter = iter(output)
    except TypeError:
        raise ValueError(f"IEFunction {name} with underlying function {func}\n"
                f"returned a value that is not an iterable\n"
                f"for input {input} -> {output}")

def map_iter(df,name,func,in_schema,out_schema,in_arity,out_arity,**kwargs):
    """helper function returns an iterator that applies a function to each row of a dataframe
    """
    for _,in_row in df.iterrows():
        in_row = list(in_row)
        assert_ie_schema(name,func,in_row,in_schema,in_arity,input_or_output='input')
        output = func(*in_row)
        assert_iterable(name,func,in_row,output)
        for out_row in output:
            out_row = coerce_tuple_like(name,func,in_row,out_row)
            out_row = list(out_row)
            assert_ie_schema(name,func,out_row,out_schema,out_arity,input_or_output='output')
            yield in_row + out_row

def ie_map(df,name,func,in_schema,out_schema,in_arity,out_arity,**kwargs):
    """given an indexed dataframe, apply an ie function to each row and return the output 
    such that each output relation is indexed by the same index as the input relation that generated it
    """
    if df is None or df.empty:
        return pd.DataFrame(columns=_col_names(in_arity+out_arity))
    output_iter = map_iter(df,name,func,in_schema,out_schema,in_arity,out_arity)
    total_arity = in_arity + out_arity
    return pd.DataFrame(output_iter,columns=_col_names(total_arity))

In [746]:

def get_rel(rel,db,**kwargs):
    # helper function to get the relation from the db for external relations
    return db[rel]

op_to_func = {
    'union':union,
    'intersection':intersection,
    'difference':difference,
    'select':select,
    'project':project,
    'rename':rename,
    'join':join,
    'ie_map':ie_map,
    'get_rel':get_rel,
    'get_const':get_const,
    'product':product,
    'groupby':groupby
}

In [747]:
def _in_cycle(g):
    return list(set(
        itertools.chain.from_iterable(nx.cycles.simple_cycles(g))
    ))

def _depends_on_cycle(g):
    in_cycle_nodes = _in_cycle(g)
    depends_on_cycle = {
        node for node in g.nodes if node in in_cycle_nodes or 
        len(set(nx.descendants(g,node)).intersection(in_cycle_nodes))>0
    }
    return depends_on_cycle

In [748]:
def calculate_dfs_rows(*dfs):
    if dfs:
        return sum([df.shape[0] for df in dfs])
    return 0

In [749]:
def profile_wrapper(op_func, profile_data, children_results, u_data):
    start = time.time()
    res = op_func(*children_results, **u_data)
    end = time.time()
    profile_data[op_func.__name__]["row_count"] += calculate_dfs_rows(*children_results)
    profile_data[op_func.__name__]["total_time"] += end - start
    return res, end - start

In [750]:
def _collect_children_and_run(G,u,results,profile_data,stack,log=False):
    children = list(G.successors(u))
    u_data = G.nodes[u]

    children_results = [results[v][-1] for v in children]
    op_func = op_to_func[u_data['op']]

    if log:
        logger.debug(f"computing node {u} with children {children} and data {u_data} , stack = {stack}")
        logger.debug(f"children results are {children_results}")
        logger.debug(f"children_data is {[G.nodes[v] for v in children]}")
    try:
        res, op_time = profile_wrapper(op_func, profile_data, children_results, u_data)
    except Exception as e:
        raise Exception(f'During excution of node {u} with args {children_results} and kwargs {u_data}'
                        f' got error {e}'
        )
    if log:
        logger.debug(f"result of node {u} is {res}")
    results[u].append(res)
    G.nodes[u]['op_time'] = op_time
    return res


In [ ]:
def compute_node(G,root,ret_inter=False,log=False):

    # makes sure there is always a last value in the list for each key
    # which is None
    list_with_none_factory = lambda : [None]
    results_dict = defaultdict(list_with_none_factory)
    profile_data = defaultdict(lambda: {"row_count": 0, "total_time": 0.0})
    start_time = time.time()
    depends_on_cycle = _depends_on_cycle(G)
    not_depends_on_cycle = [u for u in G.nodes if u not in depends_on_cycle]

    # compute non cyclic nodes in postorder
    topological_sort = list(nx.topological_sort(nx
                                                          .DiGraph(nx.subgraph(G,G.nodes))))
    for u in topological_sort[::-1]:
        res = _collect_children_and_run(G,u,results_dict,profile_data,[],log=log)
    

    end_time = time.time()
    profile_data['total_time'] = end_time - start_time
    if ret_inter:
        return res,profile_data, results_dict
    else:
        return res, profile_data


## Query ##

The query demonstrated in this notebook runs on a database of SMS messages, aiming to identify spam messages. The primary focus is on spam that falsely claims the recipient is eligible for a tax refund. The query is divided into five subqueries, each targeting a specific aspect of spam detection. All subqueries leverage regular expression-based (regex) functions for pattern matching.

I. **Year-Related Spam**  
   Detects messages containing phrases such as "refund for <year_number>" or "<year_number> open for refund," indicating potential fraud involving tax refunds for a specific year.

II. **Link-Related Spam**  
   Identifies messages with links containing keywords like "tax" or "refund," often used in spam links to deceive recipients.

III. **Money-Amount Related Spam**  
   Flags messages containing numeric patterns that resemble monetary amounts, frequently used in spam to lure victims.

IV. **Wrong Receiver**  
   Captures messages that include a "nickname" from a predefined list of known nicknames (e.g., sourced from applications like Truecaller), suggesting the message may be mistakenly or fraudulently addressed.

V. **Suspicious Sender**  
   Identifies messages sent from unknown or suspicious numbers, which are often associated with spam campaigns.


In [752]:
from spannerlib.ie_func.basic import rgx

In [753]:
magic_session = get_magic_session()

In [754]:

def rgx_is_match(pattern, # the delimeter pattern to split on
    text, # the text to be split, can be either string or Span
    ):
    """
    An IE function which given a delimeter rgx pattern and a text, 
    returns True if any match is found, False otherwise.
    """
    for _ in rgx(pattern,text):
        return [("True", )]
    return []

magic_session.register('rgx_is_match',rgx_is_match,in_schema=[str,str],out_schema=[str])

In [755]:
magic_session.import_rel('sms_rel','sms_rel.csv')

## Subquery I - tax year ##

In [756]:
%%spannerlog
sms_tax_year(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam)<-
    sms_rel(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam),
    rgx_is_match('refund for\s*([1-2][0-9]{3})',sms_body)->(res).

sms_tax_year(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam)<-
    sms_rel(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam),
    rgx_is_match('([1-2][0-9]{3})[a-zA-z ]*open for[a-zA-Z ]*refund',sms_body)->(res).


In [758]:
graph,root = magic_session.export('?sms_tax_year(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam)',plan_query=True,draw_query=True)

In [759]:
res, profile_data = compute_node(graph,root)

# Relational Optimizations

In [760]:
# We start by removing all unnecesary project nodes - all projects here that are not childs of an product nodes are redundant
def is_not_product(match,node):
    if 'product' in match[node]['op']:
        return False
    return True
            
rewrite(graph, lhs='project_node[op="project", schema];x[op]->project_node->y', p='x[op],y', rhs='x[op]->y',condition=lambda match:
    is_not_product(match, "x") and is_not_product(match, "y"), is_recursive=True)

draw(graph)

In [761]:
# Every rename's schema can be propagated to the child node
def schema(match, node):
    return match[node]['schema']
rewrite(graph, lhs='y[op="rename",schema]->z[schema];x->y', p='x,z[schema]', rhs='x->z[schema={{y_schema}}]',
        render_rhs={'y_schema': lambda match: schema(match, 'y')}, is_recursive=True)
draw(graph)

In [762]:
# move the union to be before the join,join node, and merge both projects that project sms_body into a single project node
for match in rewrite_iter(graph,
        lhs='union_node[op="union",schema]->join_node[op="join",schema]->ie_node[func,schema],join_node->sms_rel',
        p='union_node[op],ie_node[func,schema],sms_rel',
        rhs='union_node[op,schema={{ie_node_schema}}]->ie_node[func,schema], sms_rel', 
        render_rhs={'ie_node_schema': lambda match: schema(match, 'ie_node'), 'join_node_schema': lambda match: schema(match, 'join_node')}):
        join_schema = match['join_node']['schema']
rewrite(graph, lhs='project_node[op="project"]->union_node[op="union"], sms_rel[op="get_rel",schema]', 
        p='project_node[op],sms_rel[op,schema],union_node[op]',
        rhs='project_node[op]->join_node[op="join",schema={{join_schema}}]->union_node[op], join_node->sms_rel[op,schema]',
        render_rhs={'join_schema': lambda match: join_schema})
rewrite(graph, lhs='x->project_node_1[op="project"]->sms_rel[op="get_rel"], y->project_node_2[op="project"]->sms_rel', 
        p='project_node_1[op],project_node_2[op],sms_rel[op],x,y',
        rhs='x->project_node_1&project_node_2[op]->sms_rel[op],y->project_node_1&project_node_2', is_recursive=True)
draw(graph)

In [763]:
sms_rel = pd.read_csv('sms_rel.csv')
graph.nodes['sms_rel']['db'] = DB({'sms_rel': sms_rel, 'sms_tax_year': pd.DataFrame()})
before_optimization_profile_df = pd.DataFrame(profile_data).T
print("Before optimization:")
print(before_optimization_profile_df)
res, profile_data = compute_node(graph,root)
profile_df = pd.DataFrame(profile_data).T
print("After optimization:")
print(profile_df)
res

Before optimization:
              row_count  total_time
get_rel        0.000000    0.000002
rename       634.000000    0.001234
project     1794.000000    0.007870
get_const      0.000000    0.000443
product      582.000000    0.010635
ie_map       580.000000    0.067684
join         598.000000    0.003860
union         18.000000    0.001212
total_time     0.093933    0.093933
After optimization:
             row_count  total_time
get_rel       0.000000    0.000011
project     885.000000    0.002310
get_const     0.000000    0.000404
product     580.000000    0.004753
ie_map      578.000000    0.045264
union        18.000000    0.000841
join        307.000000    0.003221
total_time    0.057413    0.057413


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S234567A890127,072-3491746,False,Check if you qualify for a tax refund for 2023...,"07/02/2024, 08:25",True
1,S234567A890124,MyReTax,False,Check your eligibility for a tax refund for 20...,"21/05/2022, 20:39",True
2,S789012A345676,Missim,False,You might also receive such a message! Hello S...,"06/08/2024, 15:52",True
3,S234567A890121,TaxReturn,False,"Nadav RD 15, According to our records, you are...","22/01/2023, 19:57",True
4,S678901A234565,033-030413,False,This is the last chance to withdraw. If you ha...,"10/09/2024, 10:34",True
5,S567890A123457,FreeMoney,False,Nadav CS the tax refund for 2016 is about to e...,"31/10/2022, 13:14",True
6,S223456A787614,072-3491746,False,Your tax refund for 2018 may be higher than yo...,"05/01/2024, 16:00",True
7,S901234A567898,Missim,False,You might also receive such a message! Hello S...,"22/08/2024, 14:06",True
8,S890123A456787,TaxBack,False,We noticed that you might be eligible for a ta...,"28/08/2024, 10:07",True
9,S789012A345677,073-3489675,False,"Hi Stav Compi, Information shows you have a ta...","28/11/2023, 13:55",True


# Optimization I - subregex

In [764]:
year_dict = {"_C0":'[1-2][0-9]{3}'}

rewrite(graph, lhs='project_node[op="project"];product_node[op="product"]->project_node',
        p='product_node[op],project_node[op]',
        rhs='new_ie_node[op="ie_map",func={{func}},in_arity=2,out_arity=1,schema={{ie_schema}},name="rgx_is_match",in_schema={{in_schema}},out_schema={{out_schema}}]\
            ,new_product_node->new_get_const_node[op="get_const",const_dict={{const_dict}},schema={{const_schema}}],\
            new_product_node[op="product",schema={{product_schema}}]->project_node[op],\
            new_project_node_2[op="project",schema={{project_schema_2}}]->new_rename_node[op="rename",schema={{ie_schema}}]->new_ie_node\
            ->new_project_node_1[op="project",schema={{project_schema_1}}]->new_product_node,\
            product_node[op]->new_project_node_2',
        render_rhs={'func': lambda match: rgx_is_match,
                    'ie_schema': lambda match: ['_F0', 'sms_body', 'res'],
                    'product_schema': lambda match: ['sms_body', '_C0'],
                    'const_schema': lambda match: ['_C0'],
                    'project_schema_1': lambda match: ['_C0', 'sms_body'],
                    'project_schema_2': lambda match: ['sms_body'],
                    'const_dict': lambda match: year_dict,
                    'in_schema': lambda match: [str,str],
                    'out_schema': lambda match: [str]})

draw(graph)

In [765]:
graph.nodes['sms_rel']['db'] = DB({'sms_rel': sms_rel, 'sms_tax_year': pd.DataFrame()})
print("Profile data before optimization")
print(profile_df)
res, profile_data = compute_node(graph, root)
print("Profile data after optimization")
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res


Profile data before optimization
             row_count  total_time
get_rel       0.000000    0.000011
project     885.000000    0.002310
get_const     0.000000    0.000404
product     580.000000    0.004753
ie_map      578.000000    0.045264
union        18.000000    0.000841
join        307.000000    0.003221
total_time    0.057413    0.057413
Profile data after optimization
             row_count  total_time
get_rel       0.000000    0.000003
get_const     0.000000    0.001361
project     827.000000    0.004547
product     446.000000    0.006796
ie_map      443.000000    0.037804
rename       77.000000    0.000122
union        18.000000    0.000696
join        307.000000    0.001828
total_time    0.053981    0.053981


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S234567A890127,072-3491746,False,Check if you qualify for a tax refund for 2023...,"07/02/2024, 08:25",True
1,S234567A890124,MyReTax,False,Check your eligibility for a tax refund for 20...,"21/05/2022, 20:39",True
2,S789012A345676,Missim,False,You might also receive such a message! Hello S...,"06/08/2024, 15:52",True
3,S234567A890121,TaxReturn,False,"Nadav RD 15, According to our records, you are...","22/01/2023, 19:57",True
4,S678901A234565,033-030413,False,This is the last chance to withdraw. If you ha...,"10/09/2024, 10:34",True
5,S567890A123457,FreeMoney,False,Nadav CS the tax refund for 2016 is about to e...,"31/10/2022, 13:14",True
6,S223456A787614,072-3491746,False,Your tax refund for 2018 may be higher than yo...,"05/01/2024, 16:00",True
7,S901234A567898,Missim,False,You might also receive such a message! Hello S...,"22/08/2024, 14:06",True
8,S890123A456787,TaxBack,False,We noticed that you might be eligible for a ta...,"28/08/2024, 10:07",True
9,S789012A345677,073-3489675,False,"Hi Stav Compi, Information shows you have a ta...","28/11/2023, 13:55",True


# Subquery II - tax link

In [766]:
%%spannerlog
sms_tax_link(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam)<-
sms_rel(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam),
rgx_is_match('http.*[rR][eE][fF][uU][nN][dD].*\.com',sms_body)->(res).

sms_tax_link(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam)<-
sms_rel(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam),
rgx_is_match('http.*[tT][aA][xX].*\.com',sms_body)->(res).

In [767]:
graph, root = magic_session.export('?sms_tax_link(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam)', plan_query=True, draw_query=True)

In [768]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

              row_count  total_time
get_rel        0.000000    0.000003
rename       773.000000    0.001154
project     1933.000000    0.007151
get_const      0.000000    0.000738
product      582.000000    0.007068
ie_map       580.000000    0.043316
join         649.000000    0.003497
union         69.000000    0.001464
total_time     0.065393    0.065393


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S123456A789018,033-0129472,FALSE,Have you worked in the last 6 years? You might...,"16/02/2024, 17:00",TRUE
1,S409002A166504,Refunds,FALSE,"Nadav Numeric Algorithms, our records show you...","02/12/2024, 14:31",TRUE
2,S345678A901238,FreeTax,FALSE,Employees in the last 7 years are eligible for...,"08/02/2024, 10:15",TRUE
3,S678901A234569,033-0129472,FALSE,Free tax refund eligibility check available no...,"14/01/2024, 12:05",TRUE
4,S789012A345680,072-3491746,FALSE,Check if you qualify for a tax refund for year...,"15/01/2024, 14:00",TRUE
5,S234567A890125,FastRefunds,FALSE,Your eligibility for a refund is pending. Clic...,"10/01/2024, 11:20",TRUE
6,S789012A345681,TaxEasy,FALSE,Get a full refund estimation in minutes - most...,"29/01/2024, 15:45",TRUE
7,S399289A990880,TaxAlerts,FALSE,"Nadav the best partner, our records show you a...","30/11/2024, 18:01",TRUE
8,S567890A123459,SmartRefund,FALSE,Discover your tax refund potential today. Clic...,"27/01/2024, 11:30",TRUE
9,S901234A567901,EasyTaxRefund,FALSE,"Stav Compi Rep, Last chance! Verify your eligi...","03/01/2024, 09:15",TRUE


# Subquery III - money amount

In [769]:
%%spannerlog
sms_money(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('\d{1,3},\d{3}', sms_body) -> (res).

sms_money(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('\d{1,3}[kK]', sms_body) -> (res).

In [770]:
graph, root = magic_session.export('?sms_money(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam)', plan_query=True, draw_query=True)

In [771]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

              row_count  total_time
get_rel        0.000000    0.000105
rename       698.000000    0.006853
project     1858.000000    0.019125
get_const      0.000000    0.002271
product      582.000000    0.008990
ie_map       580.000000    0.043943
join         620.000000    0.004585
union         40.000000    0.001379
total_time     0.088614    0.088614


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S409002A166504,Refunds,FALSE,"Nadav Numeric Algorithms, our records show you...","02/12/2024, 14:31",TRUE
1,S476521A098763,FastMass,FALSE,"Stav CS Technion, 4 out of 5 people in Israel ...","07/11/2024, 14:45",TRUE
2,S901234A567898,Missim,FALSE,You might also receive such a message! Hello S...,"22/08/2024, 14:06",TRUE
3,S723019X912345,KOL MATRAA,FALSE,External loans available immediately for all b...,"22/10/2024, 15:27",FALSE
4,S890123A456790,QuickRefund,FALSE,Stav Aviram - Army you may qualify for a tax r...,"02/01/2024, 11:30",TRUE
5,S678901A234571,TaxEasy,FALSE,"Stav Combinatorics Partner, your refund could ...","11/02/2024, 16:20",TRUE
6,S476521A098764,TaxOffice15,FALSE,"There might be 8,942 NIS waiting for you from ...","30/04/2024, 20:18",TRUE
7,S567890A123457,FreeMoney,FALSE,Nadav CS the tax refund for 2016 is about to e...,"31/10/2022, 13:14",TRUE
8,S345678A901233,033-030571,FALSE,"Hi Nadav army, The year 2024 has arrived, and ...","03/07/2024, 13:40",TRUE
9,S789012A345681,TaxEasy,FALSE,Get a full refund estimation in minutes - most...,"29/01/2024, 15:45",TRUE


# Subquery IV - wrong receiver

In [772]:
truecaller_receiver_names = [
        'Nadav RD 15',
        'Stav Aviram - Army',
        'Stav CS Technion',
        'Nadav army',
        'Stav Compi Rep',
        'Stav Compi',
        'Stav My Love',
        'Nadav CS Technion',
        'Stav Combinatorics Partner',
        'Nadav Combi partner',
        'Nadav CS',
        'Nadav CS Rep',
        'Nadav Hamilton',
        'Nadav Numeric Algorithms',
        'Nadav the best partner',
        'Stavi'
        ]
s = ''
for name in truecaller_receiver_names:
    s += name + '|'
s

'Nadav RD 15|Stav Aviram - Army|Stav CS Technion|Nadav army|Stav Compi Rep|Stav Compi|Stav My Love|Nadav CS Technion|Stav Combinatorics Partner|Nadav Combi partner|Nadav CS|Nadav CS Rep|Nadav Hamilton|Nadav Numeric Algorithms|Nadav the best partner|Stavi|'

In [773]:
%%spannerlog
sms_wrong_receiver(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('(?i)\\bhi\\s(Nadav RD 15|Stav Aviram - Army|Stav CS Technion|Nadav army|Stav Compi Rep|Stav Compi|Stav My Love|Nadav CS Technion|Stav Combinatorics Partner|Nadav Combi partner|Nadav CS|Nadav CS Rep|Nadav Hamilton|Nadav Numeric Algorithms|Nadav the best partner|Stavi|)\\b.*?[.!?\\n]', sms_body) -> (res).

sms_wrong_receiver(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('(?i)\\bhello\\s(Nadav RD 15|Stav Aviram - Army|Stav CS Technion|Nadav army|Stav Compi Rep|Stav Compi|Stav My Love|Nadav CS Technion|Stav Combinatorics Partner|Nadav Combi partner|Nadav CS|Nadav CS Rep|Nadav Hamilton|Nadav Numeric Algorithms|Nadav the best partner|Stavi|)\\b.*?[.!?\\n]', sms_body) -> (res).

sms_wrong_receiver(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('(?i)(Nadav RD 15|Stav Aviram - Army|Stav CS Technion|Nadav army|Stav Compi Rep|Stav Compi|Stav My Love|Nadav CS Technion|Stav Combinatorics Partner|Nadav Combi partner|Nadav CS|Nadav CS Rep|Nadav Hamilton|Nadav Numeric Algorithms|Nadav the best partner|Stavi|).*?[.!?\\n]', sms_body) -> (res).

In [774]:
graph, root = magic_session.export('?sms_wrong_receiver(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam)', plan_query=True, draw_query=True)

In [775]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

              row_count  total_time
get_rel        0.000000    0.000087
rename      1725.000000    0.013139
project     3465.000000    0.015742
get_const      0.000000    0.002667
product      873.000000    0.014945
ie_map       870.000000    0.071312
join        1155.000000    0.005560
union        285.000000    0.003097
total_time     0.131189    0.131189


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S123456A789018,033-0129472,FALSE,Have you worked in the last 6 years? You might...,"16/02/2024, 17:00",TRUE
1,S409002A166504,Refunds,FALSE,"Nadav Numeric Algorithms, our records show you...","02/12/2024, 14:31",TRUE
2,S345678A901238,FreeTax,FALSE,Employees in the last 7 years are eligible for...,"08/02/2024, 10:15",TRUE
3,S443432A236875,Dani,TRUE,Youre not going to believe thiscall me when yo...,"05/01/2023, 20:28",FALSE
4,S934871X098345,ElectroDeals,FALSE,Hi Nadav CS Rep! Enjoy up to 50 off on top ele...,"02/12/2024, 14:30",FALSE
...,...,...,...,...,...,...
280,S789012A345682,FastRefunds,FALSE,"Hello Stav Combinatorics Partner, our records ...","12/02/2024, 09:05",TRUE
281,S456789A012343,TaxReturn,FALSE,"Stav Aviram - Army, According to our records, ...","22/01/2023, 11:32",TRUE
282,S399289A990881,HopeForIsrael,TRUE,What would you do to thank our soldiers? 1. Pr...,"29/11/2024, 12:17",FALSE
283,S970604A859824,Ahuva,FALSE,"Hi Nadav, just wanted to say youre awesome! Ha...","04/08/2024, 08:31",FALSE


# Subquery V - suspicious sender

In [776]:
%%spannerlog
sms_suspicious_sender(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('\d{3}-\d+', sender_id) -> (res).

sms_suspicious_sender(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('[tT][aA][xX]', sender_id) -> (res).

sms_suspicious_sender(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('[rR][eE][fF][uU][nN][dD]', sender_id) -> (res).

In [777]:
graph, root = magic_session.export('?sms_suspicious_sender(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam)', plan_query=True, draw_query=True)

In [778]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

              row_count  total_time
get_rel        0.000000    0.000002
rename      1181.000000    0.001658
project     3187.000000    0.013899
get_const      0.000000    0.001042
product      873.000000    0.007675
ie_map       870.000000    0.063227
join         976.000000    0.005936
union        372.000000    0.004205
total_time     0.099015    0.099015


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S123456A789018,033-0129472,FALSE,Have you worked in the last 6 years? You might...,"16/02/2024, 17:00",TRUE
1,S409002A166504,Refunds,FALSE,"Nadav Numeric Algorithms, our records show you...","02/12/2024, 14:31",TRUE
2,S216873A278941,053-4537608,FALSE,"Stav combi partner, our records show you are e...","29/11/2024, 11:46",TRUE
3,S345678A901238,FreeTax,FALSE,Employees in the last 7 years are eligible for...,"08/02/2024, 10:15",TRUE
4,S123456A789010,TaxReturn,FALSE,"Nadav RD 15, According to our records, you are...","05/01/2023, 20:27",TRUE
...,...,...,...,...,...,...
94,S234567A890121,TaxReturn,FALSE,"Nadav RD 15, According to our records, you are...","22/01/2023, 19:57",TRUE
95,S789012A345682,FastRefunds,FALSE,"Hello Stav Combinatorics Partner, our records ...","12/02/2024, 09:05",TRUE
96,S456789A012343,TaxReturn,FALSE,"Stav Aviram - Army, According to our records, ...","22/01/2023, 11:32",TRUE
97,S456789A012344,Shevah-Tax,FALSE,"Hello Stav Compi Rep, property owners that hav...","11/08/2024, 10:32",TRUE


# Putting it together

In [779]:
%%spannerlog

sms_tax(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_tax_year(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam).

sms_tax(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_tax_link(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam).

sms_tax(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_money(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam).

sms_tax(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_wrong_receiver(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam).

sms_tax(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_suspicious_sender(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam).

In [780]:
graph, root = magic_session.export('?sms_tax(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam)', plan_query=True, draw_query=True)

In [781]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

               row_count  total_time
get_rel         0.000000    0.000004
rename       5296.000000    0.008030
project     13017.000000    0.035311
get_const       0.000000    0.002538
product      3492.000000    0.023746
ie_map       3480.000000    0.251772
join         3998.000000    0.018674
union        1279.000000    0.016165
total_time      0.363645    0.363645


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S123456A789018,033-0129472,FALSE,Have you worked in the last 6 years? You might...,"16/02/2024, 17:00",TRUE
1,S409002A166504,Refunds,FALSE,"Nadav Numeric Algorithms, our records show you...","02/12/2024, 14:31",TRUE
2,S345678A901238,FreeTax,FALSE,Employees in the last 7 years are eligible for...,"08/02/2024, 10:15",TRUE
3,S443432A236875,Dani,TRUE,Youre not going to believe thiscall me when yo...,"05/01/2023, 20:28",FALSE
4,S934871X098345,ElectroDeals,FALSE,Hi Nadav CS Rep! Enjoy up to 50 off on top ele...,"02/12/2024, 14:30",FALSE
...,...,...,...,...,...,...
280,S789012A345682,FastRefunds,FALSE,"Hello Stav Combinatorics Partner, our records ...","12/02/2024, 09:05",TRUE
281,S456789A012343,TaxReturn,FALSE,"Stav Aviram - Army, According to our records, ...","22/01/2023, 11:32",TRUE
282,S399289A990881,HopeForIsrael,TRUE,What would you do to thank our soldiers? 1. Pr...,"29/11/2024, 12:17",FALSE
283,S970604A859824,Ahuva,FALSE,"Hi Nadav, just wanted to say youre awesome! Ha...","04/08/2024, 08:31",FALSE


# Relational Optimization

The graph created by spannerlog uses redundant/unnecessary relational logic. We start by optimizing the graph to use only the minimal necessary logic.

In [782]:
raw_graph = graph.copy()

The schema's column names in the graph are col_0, col_1 ... instead of the schema of the input relation. 

Notice, that any rename node's schema can be propagated to its child's schema, making the rename obsolete. Thus, we make the following rewrite:


In [783]:
rewrite(graph, lhs='y[op="rename",schema]->z[schema];x->y',
        p='x,z[schema]',
        rhs='x->z[schema={{y_schema}}]',
        render_rhs={'y_schema': lambda match: schema(match, 'y')}, is_recursive=True)
draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year, sms_tax_link, sms_money, sms_wrong_receiver, sms_suspicious_sender, sms_tax), op_time=4.0531158447265625e-06"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={0, 1, #quot;fact#quot;, 12}, op=#quot;union#quot;, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op_time=0.0010538101196289062"]
1["1
op=#quot;project#quot;, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], rule_id={0}, op_time=0.00035953521728515625"]
2["2
op=#quot;project#quot;, schem

Many projects in the graph are actually redundant - They project the entire schema of their child's node. 

In our case, we notice that the only projects that are not redundant are projects whose parents are product nodes (Since they project only the relevant columns and not the entire schema), and projects whose child is a product node (Since those projects' parents are ie_map nodes, and the project defines the order of the input columns to the ie function)

In [784]:
def is_not_product(match, node):
    if 'product' in match[node]['op']:
        return False
    return True
rewrite(graph, lhs='project_node[op="project", schema];x->project_node->y', p='x,y', rhs='x->y',condition=lambda match:
        is_not_product(match, 'x') and is_not_product(match, 'y'), is_recursive=True)

draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year, sms_tax_link, sms_money, sms_wrong_receiver, sms_suspicious_sender, sms_tax), op_time=4.0531158447265625e-06"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={0, 1, #quot;fact#quot;, 12}, op=#quot;union#quot;, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op_time=0.0010538101196289062"]
2["2
op=#quot;project#quot;, schema=[#quot;sms_body#quot;], rule_id={0}, op_time=0.0005247592926025391"]
3["3
op=#quot;get_const#quot;, const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}, schema=[#quot;_C0#quot;], rule_id={0}, op_time=0.000

IE nodes' parents are join nodes, that join the result of the ie function and sms_rel's schema. After the join operation, there is a union node for each of the subqueries. A logical optimization is making the union before the join, s.t. there is one join operation for each subquery.

In [785]:
def collection_schema(match, node):
    return match[node]['schema'][0]

rewrite(graph,
        lhs='union_node[op="union",schema];union_node->join_node[op="join",schema]->sms_rel[op="get_rel"],parent_union[op="union",schema]->union_node,join_node->ie_node[func,schema]',
        p='parent_union[op,schema],union_node[op],ie_node[func,schema],sms_rel[op]',
        rhs='parent_union[op,schema]->project_node[op="project",schema={{parent_union_schema}}]->join_node[op="join",schema={{join_node_schema}}]->union_node[op,schema={{ie_node_schema}}]->ie_node[func,schema], join_node->sms_rel[op]', 
        render_rhs={'ie_node_schema': lambda match: collection_schema(match, 'ie_node'), 'join_node_schema': lambda match: collection_schema(match, 'join_node'),
                    'parent_union_schema': lambda match: collection_schema(match, 'parent_union')},
        is_recursive=True)

draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year, sms_tax_link, sms_money, sms_wrong_receiver, sms_suspicious_sender, sms_tax), op_time=4.0531158447265625e-06"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={0, 1, #quot;fact#quot;, 12}, op=#quot;union#quot;, op_time=0.0010538101196289062, schema=[#quot;_F0#quot;, #quot;sms_body#quot;, #quot;res#quot;]"]
2["2
op=#quot;project#quot;, schema=[#quot;sms_body#quot;], rule_id={0}, op_time=0.0005247592926025391"]
3["3
op=#quot;get_const#quot;, const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}, schema=[#quot;_C0#quot;], rule_id={0}, op_time=0.00018095970153808594"]
4["4
op=#quot;product#quot;, schema=[#quot;sms_body#quot;, #quot;_C

The final relational optimization involves merging identical project nodes. In the current execution graph, each subquery contains multiple project nodes that project the same single column from the `sms_rel` schema. Specifically, four subqueries project the `sms_body` column, while the fifth subquery projects the `sender_id` column. By merging these nodes, each column is projected only once, reducing redundant operations and improving efficiency.








In [786]:

rewrite(graph, lhs='project_node[op="project",schema]->sms_rel[op="get_rel"]',
        p='sms_rel[op]',
        is_recursive=True)

rewrite(graph, lhs='sms_rel[op="get_rel"]',
        rhs='sms_rel[op],sms_body_project_node[op="project",schema={{sms_body}},sms_body]->sms_rel[op],\
                sender_id_project_node[op="project",schema={{sender_id}},sender_id]->sms_rel[op]',
        render_rhs={'sms_body': lambda match: ['sms_body'], 'sender_id': lambda match: ['sender_id']})   

rewrite(graph, lhs='sms_body[sms_body],product_node[op="product",schema]',
        rhs='product_node[op,schema]->sms_body[sms_body]',
        condition=lambda match: 'sms_body' in schema(match, 'product_node'))

rewrite(graph, lhs='sender_id[sender_id],product_node[op="product",schema]',
        rhs='product_node[op,schema]->sender_id[sender_id]',
        condition=lambda match: 'sender_id' in schema(match, 'product_node'))

draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year, sms_tax_link, sms_money, sms_wrong_receiver, sms_suspicious_sender, sms_tax), op_time=4.0531158447265625e-06"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={0, 1, #quot;fact#quot;, 12}, op=#quot;union#quot;, op_time=0.0010538101196289062, schema=[#quot;_F0#quot;, #quot;sms_body#quot;, #quot;res#quot;]"]
3["3
op=#quot;get_const#quot;, const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}, schema=[#quot;_C0#quot;], rule_id={0}, op_time=0.00018095970153808594"]
4["4
op=#quot;product#quot;, schema=[#quot;sms_body#quot;, #quot;_C0#quot;], rule_id={0}, op_time=0.0015418529510498047"]
5["5
op=#quot;project#quot;, schema=[#quot;_C0#quo

In [787]:
graph.nodes['sms_rel']['db'] = DB({'sms_rel': sms_rel, 'sms_tax_year': pd.DataFrame()})
print("Profile data before optimization")
print(profile_df)
res, profile_data = compute_node(graph, root)
print("Profile data after optimization")
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res


Profile data before optimization
               row_count  total_time
get_rel         0.000000    0.000004
rename       5296.000000    0.008030
project     13017.000000    0.035311
get_const       0.000000    0.002538
product      3492.000000    0.023746
ie_map       3480.000000    0.251772
join         3998.000000    0.018674
union        1279.000000    0.016165
total_time      0.363645    0.363645
Profile data after optimization
              row_count  total_time
get_rel        0.000000    0.000006
project     4849.000000    0.011960
get_const      0.000000    0.004339
product     3480.000000    0.035710
ie_map      3468.000000    0.270358
union       1036.000000    0.010962
join        1904.000000    0.009875
total_time     0.346269    0.346269


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S970604A859790,Bit,False,Yael Fink sent you money via bit! 45 NIS are w...,"16/04/2024, 10:28",False
1,S123456A788198,yellow,False,A new receipt has been received from Paz. To v...,"30/04/2024, 20:21",False
2,S123456A785432,yellow,False,A new receipt has been received from Paz. To v...,"22/01/2023, 19:60",False
3,S734689A482914,054-5172640,False,Important: Tax refunds for 2022 are available!...,"08/01/2024, 08:50",True
4,S127588A142811,Sarah,True,Just checking in. How have you been?,"29/11/2024, 11:47",False
...,...,...,...,...,...,...
280,S821639A531892,Boaz,True,I saw the funniest memetotally reminded me of ...,"06/11/2024, 09:28",False
281,S890123A456791,FreeTax,False,Employees in the last 7 years are eligible for...,"16/01/2024, 08:30",True
282,S409002A166504,Refunds,False,"Nadav Numeric Algorithms, our records show you...","02/12/2024, 14:31",True
283,S123456A788432,TaxOffice15,False,"Due to the situation, you are eligible for a t...","16/01/2024, 20:03",True


# Subregex optimization

This optimization targets the IE functions within each subquery. Each subquery is created using multiple regex IE functions. By identifying the largest mutual subregex among these functions and filtering rows based on it, the input dataset for each IE function is reduced, resulting in faster query execution.

For each subquery, the following structure of nodes is added for the newly introduced IE function:

`rename_node`->`ie_node`->`project_node[schema=ie_node_input_schema"]`->`product_node`->`mutual_subregex_node`, `product_node`->`project_node[schema="sms_body"/"sender_id"]`

In [788]:
relational_optimized_graph = graph.copy()

In [789]:
def largest_subregex_rewrite(g, rel, largest_subregex):
    rewrite(g, lhs='project_node[op="project"],union_node[rel="'+rel+'"];union_node->_->_->product_node[op="product"]->project_node',
        p='product_node[op],project_node[op],union_node[rel]',
        rhs='new_ie_node[op="ie_map",func={{func}},in_arity=2,out_arity=1,schema={{ie_schema}},name="rgx_is_match",in_schema={{in_schema}},out_schema={{out_schema}}],\
            new_product_node->new_get_const_node[op="get_const",const_dict={{const_dict}},schema={{const_schema}}],\
            new_product_node[op="product",schema={{product_schema}}]->project_node[op],\
            new_project_node_2[op="project",schema={{project_schema_2}}]->new_rename_node[op="rename",schema={{ie_schema}}]->new_ie_node\
            ->new_project_node_1[op="project",schema={{project_schema_1}}]->new_product_node,\
            product_node[op,rel]->new_project_node_2,\
            union_node[rel]',
        render_rhs={'func': lambda match: rgx_is_match,
                    'ie_schema': lambda match: ['_F0', 'sms_body', 'res'],
                    'product_schema': lambda match: ['sms_body', '_C0'],
                    'const_schema': lambda match: ['_C0'],
                    'project_schema_1': lambda match: ['_C0', 'sms_body'],
                    'project_schema_2': lambda match: ['sms_body'],
                    'const_dict': lambda match: largest_subregex,
                    'in_schema': lambda match: [str,str],
                    'out_schema': lambda match: [str]})

In [790]:
rel_to_largest_subregex = {'sms_tax_year': {"_C0":'[1-2][0-9]{3}'},
                           'sms_tax_link': {"_C0":'http.*\.com'},
                           'sms_money': {"_C0":'\d{1,3}'},
                           'sms_wrong_receiver': {"_C0":'Nadav RD 15|Stav Aviram - Army|Stav CS Technion|Nadav army|Stav Compi Rep|Stav Compi|Stav My Love|Nadav CS Technion|Stav Combinatorics Partner|Nadav Combi partner|Nadav CS|Nadav CS Rep|Nadav Hamilton|Nadav Numeric Algorithms|Nadav the best partner|Stavi|'}
}
for rel in rel_to_largest_subregex:
    largest_subregex_rewrite(graph, rel, rel_to_largest_subregex[rel])
draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year), op_time=6.4373016357421875e-06"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={0, 1, #quot;fact#quot;, 12}, op=#quot;union#quot;, op_time=0.0006797313690185547, schema=[#quot;_F0#quot;, #quot;sms_body#quot;, #quot;res#quot;]"]
3["3
op=#quot;get_const#quot;, const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}, schema=[#quot;_C0#quot;], rule_id={0}, op_time=0.00033211708068847656"]
4["4
op=#quot;product#quot;, schema=[#quot;sms_body#quot;, #quot;_C0#quot;], rule_id={0}, op_time=0.002193927764892578, rel=None"]
5["5
op=#quot;project#quot;, schema=[#quot;_C0#quot;, #quot;sms_body#quot;], rule_id={0}, op_time=0.000329256057739257

In [791]:
graph.nodes['sms_rel']['db'] = DB({'sms_rel': sms_rel, 'sms_tax': pd.DataFrame()})
print("Profile data before optimization:")
before_optimization_profile_df = pd.DataFrame(profile_data).T
print(before_optimization_profile_df)
res, profile_data = compute_node(graph, root)
print("Profile data after optimization:")
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

Profile data before optimization:
              row_count  total_time
get_rel        0.000000    0.000006
project     4849.000000    0.011960
get_const      0.000000    0.004339
product     3480.000000    0.035710
ie_map      3468.000000    0.270358
union       1036.000000    0.010962
join        1904.000000    0.009875
total_time     0.346269    0.346269
Profile data after optimization:
              row_count  total_time
get_rel        0.000000    0.000010
get_const      0.000000    0.004902
project     5943.000000    0.019113
product     3828.000000    0.083150
ie_map      3812.000000    0.321200
rename       750.000000    0.000413
union       1036.000000    0.011969
join        1904.000000    0.010859
total_time     0.457744    0.457744


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S970604A859790,Bit,False,Yael Fink sent you money via bit! 45 NIS are w...,"16/04/2024, 10:28",False
1,S123456A788198,yellow,False,A new receipt has been received from Paz. To v...,"30/04/2024, 20:21",False
2,S123456A785432,yellow,False,A new receipt has been received from Paz. To v...,"22/01/2023, 19:60",False
3,S734689A482914,054-5172640,False,Important: Tax refunds for 2022 are available!...,"08/01/2024, 08:50",True
4,S127588A142811,Sarah,True,Just checking in. How have you been?,"29/11/2024, 11:47",False
...,...,...,...,...,...,...
280,S821639A531892,Boaz,True,I saw the funniest memetotally reminded me of ...,"06/11/2024, 09:28",False
281,S890123A456791,FreeTax,False,Employees in the last 7 years are eligible for...,"16/01/2024, 08:30",True
282,S409002A166504,Refunds,False,"Nadav Numeric Algorithms, our records show you...","02/12/2024, 14:31",True
283,S123456A788432,TaxOffice15,False,"Due to the situation, you are eligible for a t...","16/01/2024, 20:03",True


In [792]:
subregex_optimized_graph = relational_optimized_graph.copy()
g_rel_to_largest_subregex = {'sms_tax_year': {"_C0":'[1-2][0-9]{3}'},
                            'sms_tax_link': {"_C0":'http.*\.com'},
                            'sms_money': {"_C0":'\d{1,3}[kK,]'},
#                            'sms_wrong_receiver': {"_C0":'Nadav RD 15|Stav Aviram - Army|Stav CS Technion|Nadav army|Stav Compi Rep|Stav Compi|Stav My Love|Nadav CS Technion|Stav Combinatorics Partner|Nadav Combi partner|Nadav CS|Nadav CS Rep|Nadav Hamilton|Nadav Numeric Algorithms|Nadav the best partner|Stavi|'}
}
for rel in g_rel_to_largest_subregex:
    largest_subregex_rewrite(subregex_optimized_graph, rel, rel_to_largest_subregex[rel])

In [793]:
large_sms_rel = pd.read_csv('large_sms_rel.csv')
relational_optimized_graph.nodes['sms_rel']['db'] = DB({'sms_rel':large_sms_rel, 'sms_tax': pd.DataFrame()})
subregex_optimized_graph.nodes['sms_rel']['db'] = DB({'sms_rel': large_sms_rel, 'sms_tax': pd.DataFrame()})
raw_graph.nodes['sms_rel']['db'] = DB({'sms_rel': large_sms_rel, 'sms_tax': pd.DataFrame()})
res, profile_data = compute_node(relational_optimized_graph, root)
print("Profile data before optimization:")
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res, profile_data = compute_node(subregex_optimized_graph, root)
print("Profile data after optimization:")
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

Profile data before optimization:
               row_count  total_time
get_rel          0.00000    0.000015
project     171470.00000    0.016748
get_const        0.00000    0.003025
product     131760.00000    0.062321
ie_map      131748.00000    9.572463
union        18190.00000    0.084302
join         63934.00000    0.018639
total_time       9.76231    9.762310
Profile data after optimization:
                row_count  total_time
get_rel          0.000000    0.000022
get_const        0.000000    0.002942
project     150371.000000    0.012701
product     106718.000000    0.033454
ie_map      106703.000000    7.651052
rename        3946.000000    0.000469
union        18190.000000    0.093343
join         63934.000000    0.019685
total_time       7.818929    7.818929


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S985456C789820,Batya,True,everything is fine. ;),"07/11/2023, 12:39",False
1,S359827C909859,Benjamin,True,I've got to say those books have charming titl...,"07/12/2023, 10:42",False
2,S985456G788474,Yaffa,True,Nobody cried (Cried,"09/09/2024, 15:11",False
3,S476115B098783,Foxhome,False,Flash Offer! Up to 30% off kitchenware until m...,"23/10/2022, 05:26",False
4,S234567G891037,Ron,True,Not what you'd think (;,"03/10/2021, 12:36",False
...,...,...,...,...,...,...
8664,S089011C345729,Tamar,True,but I'm not fine at all ((:,"13/04/2024, 19:53",False
8665,S234567A890966,Dani,True,couldnt believe it,"05/06/2023, 12:25",False
8666,S345678D909826,Harel,True,Cause it reminds u of innocence it smells lik...,"19/11/2024, 19:12",False
8667,S234567B891041,CinemaTLV,False,View your e-receipt here: https://cinematlv.co...,"29/11/2024, 12:16",False


In [794]:
total_time_before = 0
total_time_after_relational = 0
total_time_after_subregex = 0
for _ in range(100):
    res, profile_data = compute_node(raw_graph, root)
    total_time_before += profile_data['total_time']
    res, profile_data = compute_node(relational_optimized_graph, root)
    total_time_after_relational += profile_data['total_time']
    res, profile_data = compute_node(subregex_optimized_graph, root)
    total_time_after_subregex += profile_data['total_time']
print("Total time before optimization:", total_time_before)
print("Total time after relational optimization:", total_time_after_relational)
print("Total time after subregex optimization:", total_time_after_subregex)
print("Total time saved by relational optimization:", total_time_before - total_time_after_relational)
print("Total time saved by subregex optimization:", total_time_after_relational - total_time_after_subregex)
print("Total time saved:", total_time_before - total_time_after_subregex)
print("Average time relational optimization saved:", (total_time_before - total_time_after_relational) / 100)
print("Average time subregex optimization saved:", (total_time_after_relational - total_time_after_subregex) / 100)
print("Average total time saved:", (total_time_before - total_time_after_subregex) / 100)

Total time before optimization: 6489.436013221741
Total time after relational optimization: 5240.6468250751495
Total time after subregex optimization: 1938.269755601883
Total time saved by relational optimization: 1248.7891881465912
Total time saved by subregex optimization: 3302.3770694732666
Total time saved: 4551.166257619858
Average time relational optimization saved: 12.487891881465911
Average time subregex optimization saved: 33.02377069473267
Average total time saved: 45.511662576198574
